# YOLO Object Detection — Model

This notebook defines the model architecture. It is imported by `train.ipynb` and `main.ipynb` using `%run model.ipynb`.

In [9]:
import torch


## Conv Block

Every layer in our backbone is a `Conv` block — a standard pattern in modern CNNs:

1. **Conv2d** — learns spatial features using a sliding kernel
2. **BatchNorm2d** — normalises activations so training is stable
3. **SiLU** — activation function (smooth version of ReLU)

Padding is set to `kernel_size // 2` to preserve spatial dimensions after each convolution.

In [12]:
class Conv(torch.nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride):
        super().__init__()
        self.conv = torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding=(kernel_size//2) )
        self.batch_norm = torch.nn.BatchNorm2d(out_channels)
        self.Silu = torch.nn.SiLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.batch_norm(x)
        x = self.Silu(x)
        return x

## Architecture

We use a YOLOv2-style backbone — a series of convolution and max-pool layers that progressively downsample the image, then a detection head.

| Stage | Resolution | Purpose |
|---|---|---|
| Input | 416 × 416 | RGB image |
| Backbone | 416 → 13 × 13 | Extract features via 5 downsamples |
| Head | 13 × 13 | Predict bounding boxes |

The final layer outputs **`[170, 13, 13]`** — 13×13 grid cells, each predicting 2 boxes × 85 values (1 objectness + 4 box coords + 80 class scores).

The `Model` class reads the `architecture` list and builds `self.layers` automatically using a `layer_map` dictionary.

In [17]:
architecture = [
    # Backbone — input [3, 416, 416]
    [-1, 'Conv',    [32, 3, 1]],   # 0   416×416                                                                                                                                                     
    [-1, 'MaxPool', [2, 2]],       # 1   208×208                                                                                                                                                     
    [-1, 'Conv',    [64, 3, 1]],   # 2                                                                                                                                                               
    [-1, 'MaxPool', [2, 2]],       # 3   104×104                                                                                                                                                     
    [-1, 'Conv',    [128, 3, 1]],  # 4
    [-1, 'Conv',    [64, 1, 1]],   # 5                                                                                                                                                               
    [-1, 'Conv',    [128, 3, 1]],  # 6
    [-1, 'MaxPool', [2, 2]],       # 7   52×52                                                                                                                                                       
    [-1, 'Conv',    [256, 3, 1]],  # 8                                                                                                                                                               
    [-1, 'Conv',    [128, 1, 1]],  # 9
    [-1, 'Conv',    [256, 3, 1]],  # 10                                                                                                                                                              
    [-1, 'MaxPool', [2, 2]],       # 11  26×26
    [-1, 'Conv',    [512, 3, 1]],  # 12                                                                                                                                                              
    [-1, 'Conv',    [256, 1, 1]],  # 13
    [-1, 'Conv',    [512, 3, 1]],  # 14                                                                                                                                                              
    [-1, 'Conv',    [256, 1, 1]],  # 15                                                                                                                                                              
    [-1, 'Conv',    [512, 3, 1]],  # 16
    [-1, 'MaxPool', [2, 2]],       # 17  13×13                                                                                                                                                       
    [-1, 'Conv',    [1024, 3, 1]], # 18
    [-1, 'Conv',    [512, 1, 1]],  # 19                                                                                                                                                              
    [-1, 'Conv',    [1024, 3, 1]], # 20
    [-1, 'Conv',    [512, 1, 1]],  # 21                                                                                                                                                              
    [-1, 'Conv',    [1024, 3, 1]], # 22                                                                                                                                                              

    # Head                                                                                                                                                                                           
    [-1, 'Conv',    [1024, 3, 1]], # 23
    [-1, 'Conv',    [1024, 3, 1]], # 24                                                                                                                                                              
    [-1, 'Conv',    [170, 1, 1]],  # 25  so output shape is [170, 13, 13] so we predict 170 values for 169 cells
  ]

class Model(torch.nn.Module):
    def __init__(self, architecture=architecture):                                                                                                                                                       
        super().__init__()   # missing            
        self.architecture = architecture 
        self.layers = torch.nn.ModuleList()
        self.layer_map = {                                                                                                                                                                                   
            'Conv':      Conv,
            'MaxPool':   torch.nn.MaxPool2d,                                                                                                                                                                                                                                                                                                                               
        }
                                                                                                                                                                      
                                                                                                                                                                                                       
        in_channels = 3      
        for layer in self.architecture:                                                                                                 
            from_layer, module_type, args = layer
            module_class = self.layer_map[module_type]                                                                                                                                                       
                                                                                                                                                                                                            
            if module_type == 'Conv':                                                                                                                                                                        
                out_channels, kernel_size, stride = args                                                                                                                                                     
                self.layers.append(module_class(in_channels, out_channels, kernel_size, stride))
                in_channels = out_channels                                                                                                                                                                   
            else:
                self.layers.append(module_class(*args))
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

model = Model(architecture=architecture)
print(model)
x = torch.randn(1, 3, 416, 416)  # batch of 1 image
output = model(x)
print(output.shape)   

Model(
  (layers): ModuleList(
    (0): Conv(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (batch_norm): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (Silu): SiLU()
    )
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): Conv(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (batch_norm): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (Silu): SiLU()
    )
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv(
      (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (batch_norm): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (Silu): SiLU()
    )
    (5): Conv(
      (conv): Conv2d(128, 64, kernel_size=(1, 1), stride=(1, 1))
      (batch_norm): BatchNorm2d(64, eps=1e-05, momentu